# Cybershuttle SDK -  Molecular Dynamics
> Define, run, monitor, and analyze molecular dynamics experiments in a HPC-agnostic way.

This notebook shows how users can setup and launch a **NAMD** experiment with replicas, monitor its execution, and run analyses both during and after execution.

## Installing Required Packages

In [ ]:
%pip install -U -e "../../airavata/dev-tools/airavata-python-sdk[notebook]"

## Importing the SDK

In [1]:
import airavata_experiments as ae
import airavata_experiments.md

## Authenticating

To authenticate for remote execution, call the `ae.login()` method.
This method will prompt you to enter your credentials and authenticate your session.

In [2]:
ae.login()

Output()

Authenticated.

Once authenticated, the `ae.list_runtimes()` function can be called to list HPC resources that the user has access to.

In [3]:
runtimes = ae.list_runtimes()
ae.display(runtimes)

,cluster,category,queue_name,node_count,cpu_count,gpu_count,walltime,group
id,,,,,,,,
remote,login.expanse.sdsc.edu,gpu,gpu-shared,1,10,1,30,Default
remote,login.expanse.sdsc.edu,cpu,shared,1,10,0,30,Default
remote,anvil.rcac.purdue.edu,cpu,shared,1,24,0,30,Default
remote,login.expanse.sdsc.edu,gpu,gpu-shared,1,10,1,30,GaussianGroup
remote,login.expanse.sdsc.edu,cpu,shared,1,10,0,30,GaussianGroup
remote,anvil.rcac.purdue.edu,cpu,shared,1,24,0,30,GaussianGroup


## Uploading Experiment Files

Drag and drop experiment files onto the workspace that this notebook is run on.

```bash
(sh) $: tree data/namd
data/namd
├── b4pull.pdb
├── b4pull.restart.coor
├── b4pull.restart.vel
├── b4pull.restart.xsc
├── par_all36_water.prm
├── par_all36m_prot.prm
├── pull.conf
├── structure.pdb
└── structure.psf

1 directory, 9 files

```

## Defining a NAMD Experiment

The `ae.md.NAMD.initialize()` is used to define a NAMD experiment.
Here, provide the paths to the `.conf` file, the `.pdb` file, the `.psf` file, any optional files you want to run NAMD on.
You can preview the function definition through auto-completion.

```python
def initialize(
    name: str,
    config_file: str,
    pdb_file: str,
    psf_file: str,
    ffp_files: list[str],
    other_files: list[str] = [],
    parallelism: Literal['CPU', 'GPU'] = "CPU",
    num_replicas: int = 1
) -> Experiment[ExperimentApp]
```

To add replica runs, simply call the `exp.add_replica()` function.
You can call the `add_replica()` function as many times as you want replicas.
Any optional resource constraint can be provided here.

You can also call `ae.display()` to pretty-print the experiment.

In [ ]:
exp = ae.md.NAMD.initialize(
    name="SMD",
    config_file="data/namd/pull_gpu.conf",
    pdb_file="data/namd/structure.pdb",
    psf_file="data/namd/structure.psf",
    ffp_files=[
      "data/namd/par_all36_water.prm",
      "data/namd/par_all36m_prot.prm"
    ],
    other_files=[
      "data/namd/b4pull.pdb",
      "data/namd/b4pull.restart.coor",
      "data/namd/b4pull.restart.vel",
      "data/namd/b4pull.restart.xsc",
    ],
    parallelism="GPU",
)
for i in range(1):
    exp.create_task(*ae.list_runtimes(cluster="login.expanse.sdsc.edu", category="gpu", walltime=180))
ae.display(exp)

plan = exp.plan()
plan.save()                   # this will save the plan in DB
plan.launch()

Task created. (1 tasks in total)
Task created. (2 tasks in total)
Task created. (3 tasks in total)
Task created. (4 tasks in total)
Task created. (5 tasks in total)
Task created. (6 tasks in total)
Task created. (7 tasks in total)
Task created. (8 tasks in total)
Task created. (9 tasks in total)
Task created. (10 tasks in total)


,application,num_tasks,config_file,pdb_file,psf_file,ffp_files,parallelism,other_files,num_replicas
name,,,,,,,,,
SMD,NAMD,10,data/namd/pull_gpu.conf,data/namd/structure.pdb,data/namd/structure.psf,"data/namd/par_all36_water.prm, data/namd/par_a...",GPU,"data/namd/b4pull.pdb, data/namd/b4pull.restart...",1


Plan saved: 3203040b-5cf0-4a13-a913-3cc0a3fe2775
Preparing to launch...
Launching tasks...
[Task] Executing SMD_7F22 on Remote(args={'cluster': 'login.expanse.sdsc.edu', 'category': 'gpu', 'queue_name': 'gpu-shared', 'node_count': 1, 'cpu_count': 10, 'gpu_count': 1, 'walltime': 180, 'group': 'Default'})
[Remote] Creating Experiment: name=SMD_7F22
[AV] Preprocessing args...
[AV] Validating args...
[AV] Setting up runtime params...
[AV] Setting up application interface...
[AV] Setting up experiment...
[AV] Setting up experiment directory...
[AV] exp_dir: /Default_Project/SMD_7F22_2025_07_21_17_59_23
[AV] abs_path: /var/www/portals/gateway-user-data/cybershuttle/pjaya001@odu.edu/Default_Project/SMD_7F22_2025_07_21_17_59_23/
[AV] Setting up file inputs...
[AV] Uploading 9 file inputs for experiment...


Output()

[Remote] Failed to launch experiment: Exception("[AV] Failed to create experiment: TTransportException('TSocket read 0 bytes')")
[AV] Failed to create experiment: TTransportException('TSocket read 0 bytes')


In [ ]:
plans = ae.plan.query()
ae.display(plans)
plan.wait_for_completion()                                # wait for plan to complete

## Creating an Execution Plan

Call the `exp.plan()` function to transform the experiment definition + replicas into a stateful execution plan.

In [ ]:
plan = exp.plan()
ae.display(plan)

## Saving the Plan

A created plan can be saved locally (in JSON) or remotely (in a user-local DB) for later reference.

In [ ]:
plan.save()                   # this will save the plan in DB
plan.save_json("plan_gpu.json")   # save the plan state locally

## Launching the Plan

A created plan can be launched using the `plan.launch()` function.
Changes to plan states will be automatically saved onto the remote.
However, plan state can also be tracked locally by invoking `plan.save_json()`.

In [ ]:
plan.launch()
plan.save_json("plan_gpu.json")

## Checking the Plan Status
The status of a plan can be retrieved by calling `plan.status()`.

In [ ]:
plan.status()

## Loading a Saved Plan

A saved plan can be loaded by calling `ae.plan.load_json(plan_path)` (for local plans) or `ae.plan.load(plan_id)` (for remote plans).

In [ ]:
plan = ae.plan.load_json("plan_gpu.json")
plan = ae.plan.load(plan.id)
plan.status()
ae.display(plan)

## Fetching User-Defined Plans

The `ae.plan.query()` function retrieves all plans stored in the remote.

In [ ]:
plans = ae.plan.query()
ae.display(plans)

## Managing Plan Execution

The `plan.stop()` function will stop a currently executing plan.
The `plan.wait_for_completion()` function would block until the plan finishes executing.

In [ ]:
# plan.stop()
# plan.wait_for_completion()

## Interacting with Files

The `task` object has several helper functions to perform file operations within its context.

* `task.ls()` - list all remote files (inputs, outputs, logs, etc.)
* `task.upload(<local_path>, <remote_path>)` - upload a local file to remote
* `task.cat(<remote_path>)` - displays contents of a remote file
* `task.download(<remote_path>, <local_path>)` - fetch a remote file to local

In [ ]:
for task in plan.tasks:
    print(task.name, task.pid)
    display(task.ls())                                    # list files
    task.upload("data/sample.txt")                        # upload sample.txt
    display(task.ls())                                    # list files AFTER upload
    display(task.cat("sample.txt"))                       # preview sample.txt
    task.download("sample.txt", f"./results_{task.name}") # download sample.txt

In [ ]:
# plan.wait_for_completion()                                # wait for plan to complete
# for task in plan.tasks:
#   task.download_all(f"./results_{task.name}")             # download plan outputs


### iterate through the tasks and list the word count a file (namd_output.log) and generate a table with the results
#################### - bash command case
#################### - post-analysis script case ????

In [ ]:
plan = ae.load_plan("plan_gpu.json")

paths = []
for index, task in enumerate(plan.tasks):
    print(task)
    task.download("namd.log", f"./results_{task.name}")

In [ ]:
%%bash

ls -l results_*.log | xargs wc -l

## Executing Task-Local Code Remotely

The `@task.context()` decorator can be applied on Python functions to run them remotely within the task context.
The functions executed this way has access to the task files, as well as the remote compute resources.

**NOTE**: Currently, remote code execution is only supported for ongoing tasks. In future updates, we will support both ongoing and completed tasks. Stay tuned!

In [ ]:
# launch interactive job where we ran the N replica jobs
%request_runtime post_analysis --file=cybershuttle.yml --walltime=60 --use=expanse:shared
%wait_for_runtime post_analysis --live
%switch_runtime post_analysis

In [ ]:
%copy_data local:plan_gpu.json remote:post_analysis:plan_gpu.json

In [ ]:
import airavata_experiments as ae

plan = ae.load_plan("plan_gpu.json")

for index, task in enumerate(plan.tasks):
    print(task)
    
    @task.context
    def analyze() -> None:
        import numpy as np
        with open("pull_gpu.conf", "r") as f:
            data = f.read()
        print("pull_gpu.conf has", len(data), "chars")
        print(np.arange(10))
    analyze()